In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

def scrape_newme(url):
    print(f"Scraping URL: {url} using Selenium...")
    
    driver = webdriver.Chrome()
    try:
        driver.get(url)
        print("Waiting for page to load...")
        time.sleep(5)
        
        html = driver.page_source
        soup = BeautifulSoup(html, "html.parser")
        
        print("\n=== SEARCHING FOR PRODUCTS ===")
        
        # Find product links
        product_links = soup.select("a[href*='/product']")
        print(f"Found {len(product_links)} product links")
        
        products = []
        for i, link in enumerate(product_links[:30]):
            try:
                # Get the product href
                url_link = link.get('href', '')
                if not url_link or 'product' not in url_link.lower():
                    continue
                
                # Make sure it's a full URL
                if url_link.startswith('/'):
                    url_link = 'https://newme.asia' + url_link
                
                # Get image
                img = link.find('img')
                image_url = img.get('src') or img.get('href') if img else 'N/A'
                
                # Get title from image alt or from text
                title = ''
                if img:
                    title = img.get('alt', '')
                if not title:
                    title = link.get_text().strip()
                    # Clean up title - remove ratings and extra text
                    title = re.sub(r'^[★\d.]*', '', title).strip()
                    title = re.sub(r'Best Price:.*', '', title).strip()
                    title = re.sub(r'Delivery.*', '', title).strip()
                
                # Extract price - look for rupee symbol
                link_text = link.get_text()
                price_match = re.search(r'₹([\d,]+)', link_text)
                price = price_match.group(0) if price_match else 'N/A'
                
                # Only add if we have title
                if title and len(title) > 3:
                    product = {
                        "title": title,
                        "price": price,
                        "image_url": image_url,
                        "url": url_link
                    }
                    products.append(product)
                    
                
            except Exception as e:
                continue
        
        return pd.DataFrame(products)
    
    finally:
        driver.quit()




In [4]:


print("\n=== RESULTS ===")


#df.to_csv("newme_products.csv", index=False)
df1 = scrape_newme("https://newme.asia/collection/toonewtoignore-homepagebanner?product_cat=&subCategory=&orderby=")
print(df1.head())


=== RESULTS ===
Scraping URL: https://newme.asia/collection/toonewtoignore-homepagebanner?product_cat=&subCategory=&orderby= using Selenium...
Waiting for page to load...

=== SEARCHING FOR PRODUCTS ===
Found 30 product links
                                  title  price  \
0        Pink Floral Printed Mini Dress   ₹999   
1  Light Blue Tube Neck Lace Mini Dress  ₹1099   
2           Blue Convertible Halter Top   ₹799   
3        White Floral V-Neck Mini Dress  ₹1299   
4       Black Ruffled Halter Mini Dress  ₹1199   

                                           image_url  \
0  https://assets.newme.asia/wp-content/uploads/2...   
1  https://assets.newme.asia/wp-content/uploads/2...   
2  https://assets.newme.asia/wp-content/uploads/2...   
3  https://assets.newme.asia/wp-content/uploads/2...   
4  https://assets.newme.asia/wp-content/uploads/2...   

                                                 url  
0  https://newme.asia/product/pink-floral-printed...  
1  https://newme.asia/pro

In [5]:
print(len(df1))

30


In [7]:
df1.value_counts("title").head(20)

title
Pink Floral Printed Mini Dress                1
Light Blue Tube Neck Lace Mini Dress          1
Blue Convertible Halter Top                   1
White Floral V-Neck Mini Dress                1
Black Ruffled Halter Mini Dress               1
Navy Polka Dot Convertible Neck Mini Dress    1
Blue Solid High Rise Denim Shorts             1
White Floral Square Neck Midi Dress           1
Light Blue Polka Dot Wrap Skirt               1
Red Lace Overlay Tube Mini Dress              1
Light Blue Sweetheart Floral Maxi Dress       1
Light Pink Ruffled Sweetheart Co-Ord Set      1
Yellow Halter Crop Co-Ord Set                 1
Off-White Floral Midi Dress                   1
Dark Blue Ruched Mermaid Midi Dress           1
Light Pink Striped Tube Jumpsuit              1
Black Gingham Mid Rise Ruffle Shorts          1
Beige Solid Lace Mid Rise Skirt               1
Light Blue Striped Tie-Up Co-Ord Set          1
Cream Mid Rise Floral Midi Skirt              1
Name: count, dtype: int64

In [9]:
df1.describe()

,title,price,image_url,url
count,30,30,30,30
unique,30,9,30,30
top,Pink Floral Printed Mini Dress,₹1299,https://assets.newme.asia/wp-content/uploads/2...,https://newme.asia/product/pink-floral-printed...
freq,1,9,1,1


In [12]:
# add id
df1.insert(0, "product_id", range(1, len(df1) + 1))

In [13]:
df1.shape

(30, 5)

In [ ]:
df_combined.to_csv("products_combined.csv", index=False)

In [ ]:
# import os
# import requests
# from pathlib import Path
# from time import sleep

# def download_images(csv_path, image_url_column, save_dir="product_images"):
#     """
#     Reads CSV, downloads images from URLs, saves them locally.
    
#     csv_path         — path to your CSV file
#     image_url_column — exact column name that contains image URLs
#     save_dir         — folder where images will be saved
#     """
    
#     # Create folder if it doesn't exist
#     Path(save_dir).mkdir(exist_ok=True)
    
#     df = pd.read_csv(csv_path)
    
#     success = 0
#     failed  = 0
    
#     for idx, row in df.iterrows():
#         img_url   = row[image_url_column]
#         save_path = os.path.join(save_dir, f"{idx}.jpg")
        
#         # Skip if already downloaded
#         if os.path.exists(save_path):
#             print(f"[{idx}] Already exists, skipping")
#             continue
        
#         try:
#             response = requests.get(img_url, timeout=10)
            
#             if response.status_code == 200:
#                 with open(save_path, "wb") as f:
#                     f.write(response.content)
#                 print(f"[{idx}] Downloaded — {img_url}")
#                 success += 1
#             else:
#                 print(f"[{idx}] Failed — status {response.status_code}")
#                 failed += 1
                
#         except Exception as e:
#             print(f"[{idx}] Error — {e}")
#             failed += 1
        
#         sleep(0.3)  # be polite to the server
    
#     print(f"\nDone. Success: {success} | Failed: {failed}")
    
#     # Add local image path back to dataframe
#     df["local_image_path"] = [
#         os.path.join(save_dir, f"{i}.jpg") if os.path.exists(os.path.join(save_dir, f"{i}.jpg"))
#         else None
#         for i in range(len(df))
#     ]
    
#     # Save updated CSV with local paths
#     df.to_csv(csv_path, index=False)
#     print("CSV updated with local image paths")
    
#     return df


# df = download_images(
#     csv_path         = "products_combined.csv",
#     image_url_column = "image_url",   # ← change this to your actual column name
#     save_dir         = "product_images"
# )

[0] Downloaded — https://assets.newme.asia/wp-content/uploads/2026/04/21130251db818835/NM-PRC-345-TRS-26-MAR-32381-BLUE(0).webp
[1] Downloaded — https://assets.newme.asia/wp-content/uploads/2026/03/1715592685b4dd97/NM-PRC-178-DRS-26-MAR-31596-OFFWHITE(1).webp
[2] Downloaded — https://assets.newme.asia/wp-content/uploads/2026/03/1814131481d17570/NM-PRC-391-BLS-26-MAR-31666-WHITE(1).webp
[3] Downloaded — https://assets.newme.asia/wp-content/uploads/2025/05/171834445a6b7ead/NM-IN-56-DRS-24-DEC-13851-DARKBROWN(1)_c.webp
[4] Downloaded — https://assets.newme.asia/wp-content/uploads/2025/12/20141302005069d8/NM-PRC-385-DRS-25-DEC-28872-BLACK(1).webp
[5] Downloaded — https://assets.newme.asia/wp-content/uploads/2026/01/14125603a1b4688e/NM-IN-56-DRS-25-JUL-20853-MAROON(1).webp
[6] Downloaded — https://assets.newme.asia/wp-content/uploads/2025/11/13165048d0dd9835/NM-PRC-119-DRS-25-NOV-26550-MAROON(1).webp
[7] Downloaded — https://assets.newme.asia/wp-content/uploads/2025/11/14143607fdad3cdf/NM-P

In [1]:
import json
import pandas as pd

df = pd.read_csv("tagged_catalog.csv")
# See what color scores actually look like
print(df["color_scores"].iloc[0])
print(df["color_scores"].iloc[1])

{"soft muted pastel colors": 0.0003356468223500997, "bright bold colors": 0.11814983189105988, "dark moody colors": 0.0030786367133259773, "neutral minimal colors": 0.8784358501434326}
{"soft muted pastel colors": 0.004885707516223192, "bright bold colors": 0.3537769019603729, "dark moody colors": 0.004307503346353769, "neutral minimal colors": 0.6370298862457275}


In [2]:
# Test the matching manually
color_scores = {"soft muted pastel colors": 0.0003, "bright bold colors": 0.118, "dark moody colors": 0.003, "neutral minimal colors": 0.878}
predicted_color = "neutral"

match = max((v for k, v in color_scores.items() if predicted_color in k.lower()), default=0.0)
print(f"Match score: {match}")  # should print 0.878

Match score: 0.878


In [3]:
df = pd.read_csv("tagged_catalog.csv")
print(df["occasion_scores"].iloc[0])

{"casual everyday wear": 0.26510679721832275, "party outfit": 0.029905088245868683, "wedding guest outfit": 0.5921543836593628, "office wear": 0.05811384692788124, "date night outfit": 0.01500438991934061, "beach vacation outfit": 0.03971540182828903}


In [15]:
df = pd.read_csv("tagged_catalog.csv")

In [16]:
df.shape

(263, 10)